In [1]:
from pyspark.sql import SparkSession # type: ignore

spark = SparkSession.builder \
    .appName("SparkCourse") \
    .master("local[*]") \
    .config("spark.sql.warehouse.dir", "/home/jovyan/work/setup/spark-warehouse") \
    .config("spark.hadoop.javax.jdo.option.ConnectionURL",
            "jdbc:derby:/home/jovyan/work/metastore_db;create=true") \
    .config("spark.hadoop.javax.jdo.option.ConnectionDriverName",
            "org.apache.derby.jdbc.EmbeddedDriver") \
    .enableHiveSupport() \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 3.5.0


In [2]:
"""
Prepare club bookings dataset for analysis

+----------+------------+---------------+-------------------+--------------+
|booking_id| member_name|  facility_name|         start_time|booking_amount|
+----------+------------+---------------+-------------------+--------------+
"""

members_df = spark.table("spark_db.members")
bookings_df = spark.table("spark_db.bookings")
facilities_df = spark.table("spark_db.facilities")

club_bookings_df = (
    bookings_df.join(facilities_df, "facid")
            .join(members_df, "memid", "left")
            .selectExpr("bookid",
                        "case when memid==0 then 'Guest Member' else concat_ws(' ', firstname, surname) end as member_name",
                        "fac_name","starttime",
                        "case when memid == 0 then slots * guestcost else slots * membercost end as booking_amount")            
)

club_bookings_df.show()

+------+------------+---------------+-------------------+--------------+
|bookid| member_name|       fac_name|          starttime|booking_amount|
+------+------------+---------------+-------------------+--------------+
|     0|Darren Smith|   Table Tennis|2022-07-03 11:00:00|             0|
|     1|Darren Smith| Massage Room 1|2022-07-03 08:00:00|            70|
|     2|Guest Member|   Squash Court|2022-07-03 18:00:00|          NULL|
|     3|Darren Smith|  Snooker Table|2022-07-03 19:00:00|             0|
|     4|Darren Smith|     Pool Table|2022-07-03 10:00:00|             0|
|     5|Darren Smith|     Pool Table|2022-07-03 15:00:00|             0|
|     6| Tracy Smith| Tennis Court 1|2022-07-04 09:00:00|            15|
|     7| Tracy Smith| Tennis Court 1|2022-07-04 15:00:00|            15|
|     8|  Tim Rownam| Massage Room 1|2022-07-04 13:30:00|            70|
|     9|Guest Member| Massage Room 1|2022-07-04 15:00:00|           160|
|    10|Guest Member| Massage Room 1|2022-07-04 17:

In [ ]:
"""
Q1. Who are the top 5 members by total booking amount?

Prepare a report as the following.

member_name     | total_booking_amount
---------------------------------------
Tim Rownam      | 6480
Tim Boothe      | 3644
Gerald Butters  | 3343
Burton Tracy    | 2953
David Jones     | 2651
"""
# Using agg
from pyspark.sql.functions import col, sum # type: ignore

result_df = club_bookings_df.filter(col("member_name") != "Guest Member")\
                            .groupBy(col("member_name"))\
                            .agg(sum(col("booking_amount")).alias("total_booking_amount"))\
                            .orderBy(col("total_booking_amount").desc())\
                            .limit(5)    # select and selectExpr cannot be applied after groupBy # member name is not needed here as agg automatically takes all the columsn in the groupBy
result_df.show()

+--------------+--------------------+
|   member_name|total_booking_amount|
+--------------+--------------------+
|    Tim Rownam|                6480|
|    Tim Boothe|                3560|
|Gerald Butters|                3280|
|  Burton Tracy|                2680|
|   David Jones|                2595|
+--------------+--------------------+



In [10]:
"""
Q2. Who are the members having total booking amount > 2500?
"""
result_df = club_bookings_df.filter(col("member_name") != "Guest Member")\
                            .groupBy(col("member_name"))\
                            .agg(sum(col("booking_amount")).alias("total_booking_amount"))\
                            .filter(col("total_booking_amount") > 2500)\
                            .orderBy(col("total_booking_amount").desc())
result_df.show()

+--------------+--------------------+
|   member_name|total_booking_amount|
+--------------+--------------------+
|    Tim Rownam|                6480|
|    Tim Boothe|                3560|
|Gerald Butters|                3280|
|  Burton Tracy|                2680|
|   David Jones|                2595|
+--------------+--------------------+



In [13]:
"""
Q3. Find member wise facility bookings for more than 2500?

+-----------+--------------+--------------------+
|member_name| facility_name|total_booking_amount|
+-----------+--------------+--------------------+
| Tim Boothe|Massage Room 1|              2660.0|
| Tim Rownam|Massage Room 1|              6160.0|
"""
result_df = club_bookings_df.filter(col("member_name") != "Guest Member")\
                            .groupBy(col("member_name"), col("fac_name").alias("facility_name"))\
                            .agg(sum(col("booking_amount")).alias("total_booking_amount"))\
                            .filter(col("total_booking_amount") > 2500)\
                            .orderBy(col("total_booking_amount").desc())
result_df.show()

+-----------+--------------+--------------------+
|member_name| facility_name|total_booking_amount|
+-----------+--------------+--------------------+
| Tim Rownam|Massage Room 1|                6160|
| Tim Boothe|Massage Room 1|                2660|
+-----------+--------------+--------------------+

